# Improved RNN Model with Full CPU Utilization

This notebook implements the improved Bidirectional LSTM model with optimizations to utilize all CPU cores for faster training.

## CPU Optimization Setup

First, we'll configure TensorFlow to use all available CPU cores. This must be done before importing TensorFlow.

In [ ]:
import os
import numpy as np
import multiprocessing

# Get the number of available CPU cores
num_cores = multiprocessing.cpu_count()
print(f"Number of CPU cores available: {num_cores}")

# Set TensorFlow to use all available cores
os.environ["TF_NUM_INTRAOP_THREADS"] = str(num_cores)
os.environ["TF_NUM_INTEROP_THREADS"] = str(num_cores)
os.environ["OMP_NUM_THREADS"] = str(num_cores)
os.environ["MKL_NUM_THREADS"] = str(num_cores)

# Now import TensorFlow
import tensorflow as tf

# Configure TensorFlow session
tf.config.threading.set_intra_op_parallelism_threads(num_cores)
tf.config.threading.set_inter_op_parallelism_threads(num_cores)
tf.config.set_soft_device_placement(True)

print(f"TensorFlow is configured to use {num_cores} CPU cores")

# Try enabling mixed precision if supported
try:
    tf.keras.mixed_precision.set_global_policy('mixed_float16')
    print("Mixed precision enabled")
except:
    print("Mixed precision not supported on this device")

In [ ]:
# Import remaining libraries
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
import re
import requests
import zipfile
import io

## Parameters

We'll adjust the batch size based on the number of CPU cores for better utilization.

In [ ]:
# Parameters
max_features = 10000  # Vocabulary size
max_len = 500  # Maximum sequence length
embedding_dim = 100  # Dimension of GloVe embeddings

# Scale batch size based on core count for better CPU utilization
batch_size = 32 * (num_cores // 4)  # Scale based on core count
batch_size = max(32, min(batch_size, 256))  # Keep between 32 and 256
print(f"Using batch size: {batch_size}")

epochs = 15  # More epochs with early stopping

## Load and Preprocess the IMDB Dataset

We'll use TensorFlow's tf.data API for more efficient data loading and processing.

In [ ]:
# Load the IMDB dataset
print("Loading IMDB dataset...")
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=max_features)

# Pad sequences to ensure uniform input size
X_train = sequence.pad_sequences(X_train, maxlen=max_len)
X_test = sequence.pad_sequences(X_test, maxlen=max_len)

# Get word index for later use
word_index = imdb.get_word_index()
reverse_word_index = {value: key for key, value in word_index.items()}

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

In [ ]:
# Create optimized TensorFlow datasets
AUTOTUNE = tf.data.experimental.AUTOTUNE

# Convert to TensorFlow Dataset
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
test_dataset = tf.data.Dataset.from_tensor_slices((X_test, y_test))

# Configure for performance
train_dataset = train_dataset.cache().shuffle(10000).batch(batch_size).prefetch(AUTOTUNE)
test_dataset = test_dataset.batch(batch_size).prefetch(AUTOTUNE)

# Create validation dataset
val_size = int(0.2 * len(X_train))
train_dataset_final = train_dataset.skip(val_size)
val_dataset = train_dataset.take(val_size)

print("TensorFlow datasets created and optimized for performance")

## Download and Load GloVe Embeddings

In [ ]:
# Function to download GloVe embeddings if not already downloaded
def download_glove_embeddings():
    glove_dir = 'glove'
    if not os.path.exists(glove_dir):
        os.makedirs(glove_dir)
    
    glove_path = os.path.join(glove_dir, 'glove.6B.100d.txt')
    if not os.path.exists(glove_path):
        print("Downloading GloVe embeddings...")
        url = "https://nlp.stanford.edu/data/glove.6B.zip"
        r = requests.get(url)
        z = zipfile.ZipFile(io.BytesIO(r.content))
        z.extractall(glove_dir)
        print("Download complete!")
    else:
        print("GloVe embeddings already downloaded.")
    
    return glove_path

In [ ]:
# Load GloVe embeddings
def load_glove_embeddings(glove_path):
    print("Loading GloVe embeddings...")
    embeddings_index = {}
    with open(glove_path, encoding='utf-8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            coefs = np.asarray(values[1:], dtype='float32')
            embeddings_index[word] = coefs
    
    print(f"Found {len(embeddings_index)} word vectors.")
    
    # Create embedding matrix
    embedding_matrix = np.zeros((max_features, embedding_dim))
    for word, i in word_index.items():
        if i < max_features:
            embedding_vector = embeddings_index.get(word)
            if embedding_vector is not None:
                embedding_matrix[i] = embedding_vector
    
    return embedding_matrix

In [ ]:
# Try to download and load GloVe embeddings
try:
    glove_path = download_glove_embeddings()
    embedding_matrix = load_glove_embeddings(glove_path)
    use_glove = True
except Exception as e:
    print(f"Error loading GloVe embeddings: {e}")
    print("Continuing without pre-trained embeddings.")
    use_glove = False

## Build the Improved Model

In [ ]:
# Build an improved model with LSTM, Bidirectional, and Dropout
print("Building model...")
model = Sequential()

# Add embedding layer (with or without pre-trained embeddings)
if use_glove:
    model.add(Embedding(max_features, embedding_dim, 
                       weights=[embedding_matrix],
                       input_length=max_len,
                       trainable=False))  # Freeze the embeddings initially
else:
    model.add(Embedding(max_features, embedding_dim, input_length=max_len))

# Add Bidirectional LSTM layers with dropout
model.add(Bidirectional(LSTM(64, return_sequences=True)))
model.add(Dropout(0.3))
model.add(Bidirectional(LSTM(32)))
model.add(Dropout(0.3))

# Output layer
model.add(Dense(1, activation='sigmoid'))

# Compile the model with a lower learning rate
optimizer = Adam(learning_rate=0.001)
model.compile(optimizer=optimizer, 
              loss='binary_crossentropy', 
              metrics=['accuracy'])

# Display model summary
model.summary()

## Train the Model with Callbacks

In [ ]:
# Set up callbacks for early stopping and learning rate reduction
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=3,
    min_lr=0.0001
)

In [ ]:
# Train the model using the optimized tf.data datasets
print("Training model with optimized CPU utilization...")
history = model.fit(
    train_dataset_final,
    epochs=epochs,
    validation_data=val_dataset,
    callbacks=[early_stopping, reduce_lr]
)

## Evaluate and Save the Model

In [ ]:
# Evaluate the model
print("Evaluating model...")
loss, accuracy = model.evaluate(test_dataset)
print(f"Test accuracy: {accuracy:.4f}")

In [ ]:
# Save the model
model.save('improved_lstm_imdb_optimized.h5')
print("Model saved as 'improved_lstm_imdb_optimized.h5'")

## Test the Model on Sample Reviews

In [ ]:
# Function for text preprocessing and prediction
def preprocess_text(text):
    # Clean the text
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)  # Remove HTML tags
    text = re.sub(r'[^\w\s]', '', text)  # Remove punctuation
    text = re.sub(r'\s+', ' ', text)  # Remove extra spaces
    
    # Tokenize and convert to sequence
    words = text.split()
    encoded_review = [word_index.get(word, 2) + 3 for word in words]  # Unknown words are mapped to 2
    padded_review = sequence.pad_sequences([encoded_review], maxlen=max_len)
    
    return padded_review

def predict_sentiment(text):
    preprocessed_text = preprocess_text(text)
    prediction = model.predict(preprocessed_text)[0][0]
    sentiment = 'Positive' if prediction > 0.5 else 'Negative'
    return sentiment, prediction

In [ ]:
# Example usage
sample_reviews = [
    "This movie was fantastic! The acting was superb and the plot was engaging.",
    "Terrible film. Complete waste of time and money. The acting was wooden.",
    "I'm not sure how I feel about this movie. It had good and bad moments."
]

print("\nTesting with sample reviews:")
for review in sample_reviews:
    sentiment, score = predict_sentiment(review)
    print(f"Review: {review}")
    print(f"Sentiment: {sentiment}")
    print(f"Score: {score:.4f}")
    print("-" * 50)

## Visualize Training History

In [ ]:
# Plot training history
try:
    import matplotlib.pyplot as plt
    
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'])
    plt.plot(history.history['val_accuracy'])
    plt.title('Model Accuracy')
    plt.ylabel('Accuracy')
    plt.xlabel('Epoch')
    plt.legend(['Train', 'Validation'], loc='lower right')
    
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'])
    plt.plot(history.history['val_loss'])
    plt.title('Model Loss')
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend(['Train', 'Validation'], loc='upper right')
    
    plt.tight_layout()
    plt.savefig('training_history_optimized.png')
    print("Training history plot saved as 'training_history_optimized.png'")
except Exception as e:
    print(f"Could not generate plot: {e}")

## CPU Utilization Summary

This notebook implements several optimizations to maximize CPU utilization:

1. **Multi-threading configuration**: Sets TensorFlow to use all available CPU cores
2. **Optimized batch size**: Scales batch size based on the number of CPU cores
3. **tf.data API**: Uses efficient data loading pipelines with prefetching and caching
4. **Mixed precision**: Attempts to use mixed precision if supported by the CPU

These optimizations should result in significantly faster training times compared to the standard implementation.